In [0]:
from pyspark.sql.functions import col, when, regexp_replace


In [0]:
df_track = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
    .option("cloudFiles.schemaLocation", f"abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/checkpoint_DimTrack") \
    .load(f"abfss://bronze@adlspotifyde.dfs.core.windows.net/DimTrack/")


In [0]:
dbutils.fs.ls("abfss://bronze@adlspotifyde.dfs.core.windows.net/")

[FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimArtist/', name='DimArtist/', size=0, modificationTime=1789139755000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimArtist_cdc/', name='DimArtist_cdc/', size=0, modificationTime=1789125741000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimDate/', name='DimDate/', size=0, modificationTime=1789139890000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimDate_cdc/', name='DimDate_cdc/', size=0, modificationTime=1789125757000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimTrack/', name='DimTrack/', size=0, modificationTime=1789139825000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimTrack_cdc/', name='DimTrack_cdc/', size=0, modificationTime=1789125747000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimUser/', name='DimUser/', size=0, modificationTime=1789139678000),
 FileInfo(path='abfss://br

In [0]:
dbutils.fs.ls("abfss://silver@adlspotifyde.dfs.core.windows.net/")

[FileInfo(path='abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/', name='checkpoint/', size=0, modificationTime=0)]

In [0]:
"""
df_track = spark.read.format("parquet") \
    .load("abfss://bronze@adlspotifyde.dfs.core.windows.net/data/DimTrack")
#"""

'\ndf_track = spark.read.format("parquet")     .load("abfss://bronze@adlspotifyde.dfs.core.windows.net/data/DimTrack")\n#'

In [0]:
#display(df_track)

In [0]:
df_track = df_track.withColumn("durationFlag",
                    when (col("duration_sec") < 100, "low") \
                   .when (col("duration_sec") < 250, "medium") \
                   .otherwise("high"))
               

In [0]:
df_track = df_track.withColumn("track_name",regexp_replace(col("track_name"),'-'," "))



In [0]:
df_track = df_track.drop("_rescued_data")

In [0]:
"""
df_track.write.format("delta") \
.mode("append") \
  .save("abfss://silver@adlspotifyde.dfs.core.windows.net/track")
"""


'\ndf_track.write.format("delta") .mode("append")   .save("abfss://silver@adlspotifyde.dfs.core.windows.net/track")\n'

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION silver_ext_location

name,url,credential_name,owner,created_by,created_at,comment
silver_ext_location,abfss://silver@adlspotifyde.dfs.core.windows.net/,adb_storage_credential,nitinkumar21038@gmail.com,nitinkumar21038@gmail.com,2026-09-15T16:21:18.722Z,null


In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

name,url,comment
bronze_ext_location,abfss://bronze@adlspotifyde.dfs.core.windows.net/,null
gold_ext_location,abfss://gold@adlspotifyde.dfs.core.windows.net/,null
silver_ext_location,abfss://silver@adlspotifyde.dfs.core.windows.net/,null


In [0]:
%sql
DESCRIBE STORAGE CREDENTIAL adb_storage_credential;

name,owner,created_at,created_by,credential,comment
adb_storage_credential,nitinkumar21038@gmail.com,2026-09-15T16:20:11.230Z,nitinkumar21038@gmail.com,"AzureManagedIdentity:connector_id:/subscriptions/b8035979-0372-4c5a-a906-b01e6b39ed48/resourceGroups/RG-SpotifyDE/providers/Microsoft.Databricks/accessConnectors/AccessConnector-spotifyde,mi_id:System-Assigned MI,credential_id:6b6f5d2c-6b1f-48c5-8039-e7fe123c2b0e",null


In [0]:
df_track.writeStream.format("delta") \
    .option("outputMode","append") \
    .option("checkpointLocation",f"abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/checkpoint_DimTrack") \
    .option("path",f"abfss://silver@adlspotifyde.dfs.core.windows.net/DimTrack/data") \
    .trigger(once=True) \
    .start()

In [0]:
display(
    spark.read
        .format("delta")
        .load("abfss://silver@adlspotifyde.dfs.core.windows.net/DimTrack/data/")
)

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag
1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,medium
2,Programmable contextually based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,medium
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,medium
4,Mandatory user facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,medium
5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,medium
6,Synchronized high level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,high
7,Total human resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,medium
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,medium
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,high
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,medium


In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotifyde_catalog.silver.track
USING DELTA
LOCATION "abfss://silver@adlspotifyde.dfs.core.windows.net/DimTrack/data/"

In [0]:
%sql
SELECT * FROM spotifyde_catalog.silver.track

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag
1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,medium
2,Programmable contextually based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,medium
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,medium
4,Mandatory user facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,medium
5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,medium
6,Synchronized high level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,high
7,Total human resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,medium
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,medium
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,high
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,medium


In [0]:
spark.sql("DESCRIBE TABLE spotifyde_catalog.silver.track")

DataFrame[col_name: string, data_type: string, comment: string]

In [0]:
display(
    spark.read
        .format("parquet")
        .load("abfss://bronze@adlspotifyde.dfs.core.windows.net/DimTrack/")
)

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at
1,Stand-alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z
2,Programmable contextually-based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z
4,Mandatory user-facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z
5,Multi-layered needs-based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z
6,Synchronized high-level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z
7,Total human-resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z
